In [1]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import numpy as np
import math
from datetime import datetime, timezone, timedelta
import os
import cv2
import glob
import pandas as pd
import re
from tqdm import tqdm
from datetime import datetime
import pickle
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import folium

START_DATE="2026-04-01"
END_DATE="2026-04-15"

BASE_PATH = "/content/drive/MyDrive/TOMTOM DATA"
DATASET_PATH = os.path.join(BASE_PATH, "Dataset")
TLI_evaulation_dir = os.path.join(BASE_PATH,f"{START_DATE}_{END_DATE}", "TLI Evaluation")
TLI_figure_dir = os.path.join(TLI_evaulation_dir, "TLI graphs")
TLI_map_dir = os.path.join(TLI_evaulation_dir, "TLI Maps")
TLI_overlay_map_dir = os.path.join(TLI_evaulation_dir, "overlay_map")

os.makedirs(TLI_figure_dir, exist_ok=True)
os.makedirs(TLI_map_dir, exist_ok=True)
os.makedirs(TLI_overlay_map_dir, exist_ok=True)

MAIN_COLORS = np.array([
    (119, 119, 119),  # #777777 Grey
    (255, 35, 35),    # #FF2323 Red
    (255, 255, 55),   # #FFFF37 Yellow
    (43, 200, 43)     # #2BC82B Green

], dtype=np.int32)

COLOR_WEIGHTS = np.array([0.005, # Grey (Complete Congestion)
                          0.405, # Red (Congestion)
                          0.9,   # Yellow (Light Congestion)
                          1])    # Green (No Congestion)

COLOR_TO_WEIGHT_MAP = {
    "grey": ((119, 119, 119), 0.005),
    "red": ((255, 35, 35), 0.405),
    "yellow": ((255, 255, 55), 0.9),
    "green": ((43, 200, 43), 1)
}

CITY_CORD_DICT = {
    "Riyadh": {"lat": 24.7136, "lon": 46.6753},
    "Jeddah": {"lat": 21.5294, "lon": 39.1611},
    "Dammam": {"lat": 26.4241, "lon": 50.0905},
    "Al khobar": {"lat": 26.2199, "lon": 50.1932},
    "Dhahran": {"lat": 26.2381, "lon": 50.0430},
    "Al Qatif": {"lat": 26.5781, "lon": 49.9985},
}

# Load & Preprocess Data

In [2]:
def to_datetime_safe(x):
    DEFAULT_TZ = timezone(timedelta(hours=3))  # +03:00

    if x is None:
        return None

    if isinstance(x, datetime):
        dt = x
    else:
        dt = datetime.fromisoformat(x)

    # If naive → attach timezone
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=DEFAULT_TZ)

    return dt

def parse_filename(path):
    name = os.path.basename(path).replace(".png", "")
    parts = name.split("_")

    raw_ts = parts[7]

    # Fix time separators using regex
    fixed_ts = re.sub(
        r"T(\d{2})-(\d{2})-(\d{2})",
        r"T\1:\2:\3",
        raw_ts
    )

    fixed_ts = re.sub(
        r"\+(\d{2})-(\d{2})",
        r"+\1:\2",
        fixed_ts
    )


    metadata = {
        "city": parts[1],
        "index": int(parts[2]),
        "latitude": float(parts[3]),
        "longitude": float(parts[4]),
        "zoom": int(parts[5][1:]),     # remove 'z'
        "radius": int(parts[6][1:]),   # remove 'r'
        "timestamp": datetime.fromisoformat(fixed_ts)
    }

    return metadata

def load_city_images_paths(base_dir, start_date=None, end_date=None):
    city_dict = {}
    start_date = to_datetime_safe(start_date)
    end_date = f"{end_date}T23:59" if end_date else None
    end_date = to_datetime_safe(end_date)

    for city_name in os.listdir(base_dir):
        city_path = os.path.join(base_dir, city_name)

        if not os.path.isdir(city_path):
            continue

        city_map_path = os.path.join(city_path, "stitched")
        if not os.path.isdir(city_map_path):
            continue

        image_paths = glob.glob(os.path.join(city_map_path, "*.png"))

        filtered_paths = []

        for path in image_paths:
            meta = parse_filename(path)
            ts = meta["timestamp"]

            if start_date and ts < start_date:
                continue
            if end_date and ts > end_date:
                continue

            filtered_paths.append(path)

        city_dict[city_name] = filtered_paths

    return city_dict

city_dict = load_city_images_paths(DATASET_PATH, start_date=START_DATE, end_date=END_DATE)

## Visualize Unique Colors

In [ ]:
def extract_unique_colors(img):
    # Get Unique Colors
    if img.mode != 'RGB':
        img = img.convert('RGB')

    arr = np.asarray(img, dtype=np.uint8)
    pixels = arr.reshape(-1, arr.shape[-1])
    colors, counts = np.unique(pixels, axis=0, return_counts=True)

    # Convert to list of tuples
    color_counts = list(zip(map(tuple, colors), counts))
    color_counts.sort(key=lambda x: x[1], reverse=True)

    return color_counts

img_path = city_dict["Riyadh"][0]

img = Image.open(img_path)
# Get 700 most unique colors
top_k = 700
unique_colors = extract_unique_colors(img)
uniuque_colors_sample = unique_colors[:top_k]
colors_sample = np.array([color for color, count in uniuque_colors_sample])
counts_sample = np.array([count for color, count in uniuque_colors_sample])

print(f"Image loaded successfully. Image format: {img.format}, size: {img.size}, mode: {img.mode}")
display(img)

## Plot the 700 most repeated colors

In [ ]:
# Plot the 700 most repeated colors
swatch_size = 10 # Each swatch will be 5x5 pixels
num_colors = len(colors_sample)

# Calculate the dimensions of the visualization image
num_cols = min(60, num_colors) # Limit columns
num_rows = math.ceil(num_colors / num_cols)

img_width = num_cols * swatch_size
img_height = num_rows * swatch_size

print(f"Creating a color palette visualization of size {img_width}x{img_height} pixels.")
print(f"Number of columns: {num_cols}, Number of rows: {num_rows}")
print("Desendent sort as follows:")
print("-->\n-->")

# Create a blank RGB image
color_palette_img = Image.new('RGB', (img_width, img_height), color = 'white') # Start with a white background
draw = ImageDraw.Draw(color_palette_img)

# Iterate through the unique_colors list and draw swatches
for i, color in enumerate(colors_sample):
    # Calculate x and y coordinates for the top-left corner of the current swatch
    x = (i % num_cols) * swatch_size
    y = (i // num_cols) * swatch_size

    # Draw a filled rectangle (the swatch) with the current unique color
    draw.rectangle([x, y, x + swatch_size, y + swatch_size], fill=tuple(color))

# Display the generated visualization image
display(color_palette_img)

## Plot Color Variation in 3D


In [ ]:
# Plot Color Variation in 3D

# Origin colors
normalized_main_colors = MAIN_COLORS / 255.0
main_r = MAIN_COLORS[:, 0]
main_g = MAIN_COLORS[:, 1]
main_b = MAIN_COLORS[:, 2]

# Split channels
r = colors_sample[:, 0]
g = colors_sample[:, 1]
b = colors_sample[:, 2]

# Convert RGB (0-255) to matplotlib color format (0-1)
normalized_colors = colors_sample / 255.0

# Create figure
fig = plt.figure(figsize=(15, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    r, g, b,
    c=normalized_colors,
)

ax.scatter(
    main_r, main_g, main_b,
    c=normalized_main_colors,
    marker='s',
    s=200,
)

ax.set_xlabel('Red')
ax.set_ylabel('Green')
ax.set_zlabel('Blue')

ax.set_xlim(0, 255)
ax.set_ylim(0, 255)
ax.set_zlim(0, 255)

plt.title("3D RGB Color Distribution")
plt.show()

## Image Color Caliberation

In [6]:
def get_img(img_path):
    "Rteturn color calibrated image as ndarry"
    try:
        img = Image.open(img_path)
        return calibrate_colors(img)
    except FileNotFoundError:
        print(f"Error: The file '{img_path}' was not found. Please ensure you have uploaded the image to your Colab environment and that the filename is correct.")
    except Exception as e:
        print(f"An error occurred: {e}")

    return None

def calibrate_colors(img):
    "Make sure that colors follow restricted color code"
    img_rgb = img.convert("RGB")
    img_array = np.asarray(img_rgb, dtype=np.uint8)

    # Reshape image to (num_pixels, 3)
    h, w, _ = img_array.shape
    flat_pixels = img_array.reshape(-1, 3).astype(np.int16)

    # Compute squared distances to each main color
    min_dist = np.full(flat_pixels.shape[0], np.inf)
    min_idx = np.zeros(flat_pixels.shape[0], dtype=np.int32)

    for i, color in enumerate(MAIN_COLORS):
        diff = flat_pixels - color  # (N, 3)
        dist = np.sum(diff * diff, axis=1)  # (N,)

        mask = dist < min_dist
        min_dist[mask] = dist[mask]
        min_idx[mask] = i

    # Apply threshold
    threshold_sq = 150 ** 2
    mask = min_dist <= threshold_sq

    # Create output array (default black)
    new_pixels = np.zeros_like(flat_pixels)

    # Replace only pixels within threshold
    new_pixels[mask] = MAIN_COLORS[min_idx[mask]]

    # Reshape back to image
    return new_pixels.reshape(h, w, 3).astype(np.uint8)

In [ ]:
# Preprocessing - Image Color Caliberation
img_array = get_img(img_path)
Image.fromarray(img_array)

# Traffic Load Index Calculations

In [8]:
def compute_tli(image_path):
    img_array = get_img(image_path)
    if img_array is None:
        return 0

    # Flatten pixels into shape (num_pixels, 3)
    pixels = img_array.reshape(-1, 3)
    px_counts = []
    for color in MAIN_COLORS:
        mask = np.all(pixels == color, axis=1)
        px_counts.append(np.sum(mask))
    px_counts = np.array(px_counts)

    total = np.sum(px_counts)
    if total == 0:
        return 0

    return np.sum(COLOR_WEIGHTS * px_counts) / total


def get_city_tli_dict(city_dict, force_recompute=False):
    PICKLE_PATH = os.path.join(BASE_PATH, "city_TLI_Inv_dict.pkl")
    # 1. Check if pickle exists
    if os.path.exists(PICKLE_PATH) and not force_recompute:
        print("Loading cached TLI data from pickle...")
        with open(PICKLE_PATH, "rb") as f:
            return pickle.load(f)

    # 2. Otherwise compute
    city_TLI_dict = {}
    for city_name, city_path_list in city_dict.items():
        records = []
        print(f"Processing {city_name}")
        for path in tqdm(city_path_list):
            meta = parse_filename(path)
            tli = compute_tli(path)

            meta["TLI"] = tli
            meta["TLI_inv"] = 1/tli
            records.append(meta)

        city_TLI_dict[city_name] = pd.DataFrame(records)
        # 3. Save to pickle
    print("Saving results to pickle...")
    with open(PICKLE_PATH, "wb") as f:
        pickle.dump(city_TLI_dict, f)
    return city_TLI_dict

# Plot TLI_inve for Cities




In [ ]:
# Draw TLI over day hours
city_tli_dict = get_city_tli_dict(city_dict)
weekday_colors = {
    "Sat": "brown", # Saturday is yellow
    "Sun": "magenta",   # Sunday is cyan
    "Mon": "blue",   # Monday is blue
    "Tue": "orange",  # Tuesday is green
    "Wed": "green", # Wednesday is orange
    "Thu": "purple", # Thursday is purple
    "Fri": "red",    # Friday is red
}
for city_name, df in city_tli_dict.items():
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Extract date and hour
    df["date"] = df["timestamp"].dt.date
    df["hour"] = df["timestamp"].dt.hour
    df["weekday"] = df["timestamp"].dt.strftime("%a")

    # Optional: aggregate if multiple captures per hour

    df_hourly = df.groupby(["date", "hour", "weekday"])["TLI_inv"].mean().reset_index()
    img_path = os.path.join(TLI_figure_dir, f"Daily TLI_inv Pattern ({city_name})")
    used_labels = set()  # to avoid duplicate legend entries

    for date, group in df_hourly.groupby("date"):

        weekday = group["weekday"].iat[0]

        color = weekday_colors.get(weekday, "black")

        # Add label only once per weekday
        label = weekday if weekday not in used_labels else None
        used_labels.add(weekday)

        plt.plot(
            group["hour"],
            group["TLI_inv"],
            color=color,
            marker=".",
            markersize=8,
            label=label
        )

    plt.xlabel("Hour of Day")
    plt.ylabel("Traffic Load Index (TLI_inv)")
    plt.title(f"Daily TLI_inv Pattern ({city_name})")

    hours = list(range(24))
    plt.xticks(hours, [f"{h:02d}:00" for h in hours], rotation=45)
    weekday_order = [ "Sat", "Sun", "Mon", "Tue", "Wed", "Thu", "Fri"]

    handles, labels = plt.gca().get_legend_handles_labels()

    ordered = [(h, l) for l, h in sorted(zip(labels, handles), key=lambda x: weekday_order.index(x[0]))]
    handles, labels = zip(*ordered)
    plt.legend(handles, labels)
    plt.tight_layout()
    plt.savefig(img_path, bbox_inches='tight', pad_inches=0.05)  # Saves as PNG
    plt.show()

In [ ]:
# Generate weight map for Cities
def image_to_weight_map(path):
    img = get_img(path)
    if img is None:
        return None
    h, w, _ = img.shape
    weight_map = np.zeros((h, w), dtype=np.float32)

    for color, weight in zip(MAIN_COLORS, COLOR_WEIGHTS):
        mask = np.all(img == color, axis=-1)
        weight_map[mask] = weight

    return weight_map

city_tli_dict = {}
for city_name, city_path_list in city_dict.items():
    print(f"Processing {city_name}")
    running_sum = None
    count = 0

    for path in tqdm(city_path_list):
        wm = image_to_weight_map(path)
        if wm is None:
            continue
        if running_sum is None:
            running_sum = np.zeros_like(wm, dtype=np.float32)
        running_sum += wm
        count += 1
        del wm
    pixel_tli = running_sum / count
    non_zero_mask = pixel_tli > 0
    pixel_tli_inv = np.zeros_like(pixel_tli)
    pixel_tli_inv[non_zero_mask] = 1 / pixel_tli[non_zero_mask]
    city_tli_dict[city_name] = pixel_tli_inv

In [ ]:
def tli_to_image(pixel_tli):
    h, w = pixel_tli.shape
    output = np.zeros((h, w, 3), dtype=np.uint8)

    # Create masks
    balck_mask = (pixel_tli == 0)
    green_mask  = (pixel_tli >= 1.0) & (pixel_tli < 1.053)
    Yellow_mask = (pixel_tli >= 1.053) & (pixel_tli < 1.170)
    red_mask = (pixel_tli >= 1.170) & (pixel_tli < 4.820)
    grey_mask    = (pixel_tli >= 4.820)

    output[grey_mask]  = [119, 119, 119]
    output[green_mask] = [43, 200, 43]
    output[Yellow_mask] = [255, 255, 55]
    output[red_mask]    = [255, 35, 35]
    output[balck_mask]  = [0, 0, 0]

    return output

def plot_TLI_inv_map(map_img, title,
                     tight_plot=False, show_legend=True, save_plot=True):
    # --- Normalize values ---
    sorted_color_map = [
        (value[0], 1/value[1])
        for key, value in sorted(
            COLOR_TO_WEIGHT_MAP.items(),
            key=lambda item: 1 / item[1][1]  # inverse of weight
        )
    ]
    TLI_inv_list = np.array([value[1] for value in sorted_color_map])
    rgb_list = np.array([tuple(c/255 for c in value[0]) for value in sorted_color_map])

    # --- Create figure ---
    fig, ax = plt.subplots(dpi=1200)
    ax.axis("off")

    # --- Plot image ---
    ax.imshow(map_img)
    if save_plot:
        img_name = f"{title} - tight"
        save_path = os.path.join(TLI_map_dir, img_name)
        plt.savefig(save_path, bbox_inches="tight", pad_inches=0)

    ax.set_title(title, fontsize=10)
    if show_legend:
        # --- Discreet legend (optional, simplified) ---
        legend_elements = []
        for rgb, val in sorted(
            [(v[0], 1 / v[1]) for v in COLOR_TO_WEIGHT_MAP.values()],
            key=lambda x: x[1]
        ):
            color = tuple(c / 255 for c in rgb)
            legend_elements.append(
                Patch(facecolor=color, edgecolor='none', label=f"{val:.1f}")
            )

        legend = ax.legend(
            handles=legend_elements,
            title=title,
            fontsize=6,
            title_fontsize=7,
            loc="upper center",
            frameon=False,
            ncol=len(legend_elements),
            bbox_to_anchor=(0.5, -0.02)
        )

    # --- Save ---
    if save_plot:
        save_path = os.path.join(TLI_map_dir, title)
        plt.savefig(save_path, bbox_inches="tight", pad_inches=0.05)
    plt.show()

# --- Run ---
for city_name, pixel_tli_inv in city_tli_dict.items():
    print(f"Start Processing ({city_name})")
    map_img_arr = tli_to_image(pixel_tli_inv)

    plot_TLI_inv_map(
        map_img_arr,
        f"Average Congestion Intensity - {city_name}",
        tight_plot=True,
        show_legend=False
    )

# Create overlay map for TLI Congestions


In [ ]:
# Overlay image over a map
def latlon_to_tile(lat_deg: float, lon_deg: float, zoom: int):
    """
    Convert latitude/longitude to XYZ tile coordinates.
    """
    lat_rad = math.radians(lat_deg)
    n = 2.0 ** zoom
    x_tile = int((lon_deg + 180.0) / 360.0 * n)
    y_tile = int((1.0 - math.log(math.tan(lat_rad) + (1 / math.cos(lat_rad))) / math.pi) / 2.0 * n)
    return x_tile, y_tile

def tile_bounds(x: int, y: int, zoom: int, radius: int):
    """
    Return lat/lon bounds of the tile as (north, west, south, east)
    """
    n = 2.0 ** zoom
    lon_w = (x - radius) / n * 360.0 - 180.0
    lon_e = (x + radius + 1) / n * 360.0 - 180.0
    lat_n = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y - radius) / n))))
    lat_s = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + radius + 1) / n))))
    return lat_n, lon_w, lat_s, lon_e

def make_black_transparent(img_path):
    img = Image.open(img_path).convert("RGBA")
    arr = np.array(img)

    # Identify black pixels
    black_mask = np.all(arr[:, :, :3] == [0, 0, 0], axis=-1)

    # Set alpha channel
    arr[black_mask, 3] = 0      # transparent
    arr[~black_mask, 3] = 255   # fully visible

    return arr


overlay_maps_dict = {}
# Iterate over each city directory
image_path_list = glob.glob(os.path.join(TLI_map_dir, "*.png"))
tight_maps_list = [map for map in image_path_list if "tight" in map]

if len(tight_maps_list) == 0:
    print("Found no tight maps to overlay")
else:
    for img_path in tqdm(tight_maps_list, "Generating Maps"):
        match_re = re.search(r"-\s*([^-]+?)(?:\s*-\s*tight)?\.[a-zA-Z0-9]+$",
                          img_path,
                          re.IGNORECASE)
        if match_re:
            city = match_re.group(1).strip()

        if city not in overlay_maps_dict:
            overlay_maps_dict[city] = []
        overlay_maps_dict[city].append(img_path)

        city_cords = CITY_CORD_DICT[city]
        zoom = 14
        radius = 6
        x, y = latlon_to_tile(city_cords['lat'], city_cords['lon'], zoom)
        north, west, south, east = tile_bounds(x, y, zoom, radius)

        map = folium.Map(location=[city_cords['lat'], city_cords['lon']], zoom_start=zoom)
        overlay_img = make_black_transparent(img_path)
        folium.raster_layers.ImageOverlay(
            name=f"TLI Map ({city})",
            image=overlay_img,
            bounds=[[south, west], [north, east]],
            opacity=0.55,
        ).add_to(map)

        folium.LayerControl().add_to(map)

        map.save(os.path.join(TLI_overlay_map_dir,
                            f"onset_overlay_map - {city}.html"))